In [ ]:
os.makedirs(KEYPOINTS_FOLDER, exist_ok=True)

# augmentation settings (applied ONLY on TRAIN split)
AUGMENT_FOR_LESS_THAN = 3      # if class has < 3 ORIGINAL videos -> augment
AUG_CLIPS_PER_VIDEO = 4        # number of augmentations (excluding original)
TARGET_SAMPLES_PER_CLASS = 8  
DUPLICATE_TO_REACH = True

# training
MAX_EPOCHS = 50
BATCH_SIZE = 4
PATIENCE = 6
RANDOM_SEED = 42

# sequence processing
FEAT_DIM = 126
FIXED_LEN = 100 

# prediction unknown logic
PRED_THRESHOLD = 0.60
PRED_MARGIN = 0.15

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# ----------------------------
# LABEL EXTRACTION
# ----------------------------
def clean_label_from_filename(filename: str) -> str:
    base = os.path.basename(filename)
    part = base.split('-', 1)[0]
    part = re.sub(r'\(.*?\)', '', part)  # remove (SS) etc
    part = re.sub(r'[^A-Za-z0-9ÄÖÅäöåÜüÉéĄąŁłŚśĆćŻżŹź\- ]+', '', part)
    return part.strip().upper()

# ----------------------------
# AUGMENTATION
# ----------------------------
def write_clip_to_tmp(clip: VideoFileClip, path: str) -> str:
    clip.write_videofile(path, codec="libx264", audio=False, verbose=False, logger=None)
    return path

def generate_augmented_temp_paths(video_path: str):
    clip = VideoFileClip(video_path)
    tmp_dir = tempfile.gettempdir()
    base = os.path.splitext(os.path.basename(video_path))[0]

    paths = []

    # original
    orig = os.path.join(tmp_dir, f"{base}_orig.mp4")
    write_clip_to_tmp(clip, orig)
    paths.append(orig)

    # augmentations: brightness + speed
    aug_funcs = [
        lambda c: c.fx(vfx.colorx, 1.25),
        lambda c: c.fx(vfx.colorx, 0.75),
        lambda c: c.fx(vfx.speedx, 1.2),
        lambda c: c.fx(vfx.speedx, 0.85),
    ]

    for i in range(AUG_CLIPS_PER_VIDEO):
        aug = aug_funcs[i % len(aug_funcs)](clip)
        p = os.path.join(tmp_dir, f"{base}_aug_{i}.mp4")
        write_clip_to_tmp(aug, p)
        paths.append(p)

    clip.close()
    return paths

# ----------------------------
# MEDIAPIPE KEYPOINT EXTRACTION
# ----------------------------
mp_hands = mp.solutions.hands

def extract_hand_keypoints_consistent(video_path, max_frames=None):
    cap = cv2.VideoCapture(video_path)
    seq = []

    with mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as hands:
        frame_idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame_idx += 1
            if max_frames and frame_idx > max_frames:
                break

            image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = hands.process(image_rgb)

            left = np.zeros((21, 3), dtype=np.float32)
            right = np.zeros((21, 3), dtype=np.float32)

            if results.multi_hand_landmarks and results.multi_handedness:
                for lm, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
                    coords = np.array([[p.x, p.y, p.z] for p in lm.landmark], dtype=np.float32)
                    if handedness.classification[0].label.lower().startswith("left"):
                        left = coords
                    else:
                        right = coords

            seq.append(np.concatenate([left.flatten(), right.flatten()]))

    cap.release()
    return np.array(seq, dtype=np.float32)  # [T,126] (T variable)

# ----------------------------
# KEYPOINT NORMALIZATION (recommended)
# ----------------------------
def normalize_seq(seq_126):
    # seq_126: [T, 126]
    if seq_126.shape[0] == 0:
        return seq_126

    seq = seq_126.reshape((-1, 2, 21, 3))  # [T, hand, lm, xyz]
    out = np.zeros_like(seq, dtype=np.float32)

    for h in range(2):
        hand = seq[:, h, :, :]  # [T,21,3]

        wrist = hand[:, 0:1, :]  # [T,1,3]
        hand0 = hand - wrist

        scale = np.linalg.norm(hand0[:, 9, :], axis=1, keepdims=True)  # [T,1]
        scale = np.clip(scale, 1e-6, None)

        handn = hand0 / scale[:, None, :]  # [T,21,3]
        out[:, h, :, :] = handn

    return out.reshape((-1, 126)).astype(np.float32)

# ----------------------------
# FIXED LENGTH SAMPLING
# ----------------------------
def to_fixed_length(seq, T=FIXED_LEN):
    # seq: [t,126]
    t = seq.shape[0]
    if t == 0:
        return np.zeros((T, FEAT_DIM), dtype=np.float32)
    if t == T:
        return seq.astype(np.float32)
    if t > T:
        idx = np.linspace(0, t - 1, T).astype(int)
        return seq[idx].astype(np.float32)
    # t < T
    pad = np.zeros((T - t, FEAT_DIM), dtype=np.float32)
    return np.concatenate([seq.astype(np.float32), pad], axis=0)

# ----------------------------
# STEP A: LIST ORIGINALS + GROUP SPLIT FIRST
# ----------------------------
video_files = sorted([f for f in os.listdir(VIDEO_FOLDER)
                      if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))])

orig_items = [(os.path.join(VIDEO_FOLDER, f), clean_label_from_filename(f)) for f in video_files]
orig_labels = [lab for _, lab in orig_items]
counts = Counter(orig_labels)

if len(orig_items) == 0:
    raise RuntimeError(f"No videos found in {VIDEO_FOLDER}")

# stratify only if every class has >= 2 originals
can_stratify = min(counts.values()) >= 2

train_items, val_items = train_test_split(
    orig_items,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=orig_labels if can_stratify else None
)

print("Original train videos:", len(train_items))
print("Original val videos:", len(val_items))
print("Train label counts:", Counter([y for _, y in train_items]))
print("Val label counts:", Counter([y for _, y in val_items]))

# ----------------------------
# STEP B: BUILD PROCESS LISTS
# - augment only train
# - no augmentation in validation
# ----------------------------
to_process_train = []
for path, label in train_items:
    if counts[label] < AUGMENT_FOR_LESS_THAN:
        tmp_paths = generate_augmented_temp_paths(path)
        for p in tmp_paths:
            to_process_train.append((p, label))

        if DUPLICATE_TO_REACH and len(tmp_paths) < TARGET_SAMPLES_PER_CLASS:
            reps = math.ceil(TARGET_SAMPLES_PER_CLASS / len(tmp_paths))
            for _ in range(reps - 1):
                for p in tmp_paths:
                    to_process_train.append((p, label))
    else:
        to_process_train.append((path, label))

to_process_val = [(path, label) for path, label in val_items]

print("Train clips to process:", len(to_process_train))
print("Val clips to process:", len(to_process_val))

# ----------------------------
# STEP C: EXTRACT + SAVE KEYPOINTS
# (separate train/val prefixes)
# ----------------------------
def process_and_save(to_process, split_tag):
    saved = []
    for i, (path, label) in enumerate(to_process):
        base = os.path.splitext(os.path.basename(path))[0]
        safe_base = re.sub(r'[^A-Za-z0-9_.\-]+', '_', base)
        npy_name = f"{split_tag}__{label}__{safe_base}.npy"
        npy_path = os.path.join(KEYPOINTS_FOLDER, npy_name)

        if os.path.exists(npy_path) and os.path.getsize(npy_path) > 0:
            saved.append((npy_path, label))
            continue

        print(f"[{split_tag} {i+1}/{len(to_process)}] {path}")
        seq = extract_hand_keypoints_consistent(path)

        if seq.shape[0] == 0:
            print("  -> no keypoints detected, skipping")
            continue

        seq = normalize_seq(seq)
        seq = to_fixed_length(seq, FIXED_LEN)

        np.save(npy_path, seq.astype(np.float32))
        saved.append((npy_path, label))
    return saved

saved_train = process_and_save(to_process_train, "train")
saved_val = process_and_save(to_process_val, "val")

print("Saved train keypoints:", len(saved_train))
print("Saved val keypoints:", len(saved_val))

if len(saved_train) == 0 or len(saved_val) == 0:
    raise RuntimeError("Not enough extracted sequences for training/validation. Check videos/keypoints.")

# ----------------------------
# STEP D: BUILD X_train, y_train, X_val, y_val
# ----------------------------
X_train_list, y_train_list = [], []
for p, label in saved_train:
    X_train_list.append(np.load(p).astype(np.float32))
    y_train_list.append(label)

X_val_list, y_val_list = [], []
for p, label in saved_val:
    X_val_list.append(np.load(p).astype(np.float32))
    y_val_list.append(label)

# Encode labels (fit on TRAIN labels only)
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train_list)
y_val_enc = le.transform(y_val_list)

num_classes = len(le.classes_)
print("Classes:", list(le.classes_))

X_train = np.stack(X_train_list, axis=0)  # [N,T,126]
X_val = np.stack(X_val_list, axis=0)

y_train = to_categorical(y_train_enc, num_classes=num_classes).astype(np.float32)
y_val = to_categorical(y_val_enc, num_classes=num_classes).astype(np.float32)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)

# ----------------------------
# STEP E: MODEL
# ----------------------------
model = Sequential([
    Masking(mask_value=0.0, input_shape=(FIXED_LEN, FEAT_DIM)),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(num_classes, activation="softmax"),
])

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

callbacks = [
    EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True, verbose=1),
    ModelCheckpoint("best_sign_model_tf213_new.h5", monitor="val_loss", save_best_only=True, verbose=1),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    shuffle=True,
    verbose=1
)

# ----------------------------
# STEP F: SAVE MODEL + ENCODER + META
# ----------------------------
model.save("sign_model_tf213_new.h5")
with open("label_encoder_new.pkl", "wb") as f:
    pickle.dump(le, f)

meta = {
    "fixed_len": FIXED_LEN,
    "feat_dim": FEAT_DIM,
    "pred_threshold": PRED_THRESHOLD,
    "pred_margin": PRED_MARGIN
}
with open("model_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("Saved: sign_model_tf213_new.h5, label_encoder_new.pkl, model_meta.json")

# ----------------------------
# STEP G: PREDICT FUNCTION
# ----------------------------
def predict_video(video_path, threshold=PRED_THRESHOLD, margin=PRED_MARGIN):
    seq = extract_hand_keypoints_consistent(video_path)
    if seq.shape[0] == 0:
        return ("NoKeypoints", 0.0)

    seq = normalize_seq(seq)
    seq = to_fixed_length(seq, FIXED_LEN)

    inp = np.expand_dims(seq, axis=0).astype(np.float32)
    probs = model.predict(inp, verbose=0)[0]

    top1 = int(np.argmax(probs))
    top1_conf = float(probs[top1])

    # top2
    top2 = int(np.argsort(probs)[-2])
    top2_conf = float(probs[top2])

    label = le.inverse_transform([top1])[0]

    if (top1_conf < threshold) or ((top1_conf - top2_conf) < margin):
        return ("Unknown", top1_conf)

